# Gold — agregados: scorecard, SLA, KPIs diários e atraso × review (etapa 4.3)

Segunda fonte do pipeline `olist.gold`: **não substitui** o `01_gold_modelo`, lê as tabelas dele e declara quatro agregados prontos para consumo (dashboard, Genie, SQL):

| Tabela | Granularidade | Responde |
|---|---|---|
| `gold_seller_scorecard` | 1 linha por **versão do cadastro** do vendedor | como cada vendedor performa em GMV, cancelamento, entrega no prazo e review, **com a cidade e a UF que ele tinha na data da venda** |
| `gold_delivery_sla` | mês da compra × UF do cliente | entrega no prazo, atraso e lead time por região |
| `gold_daily_kpis` | dia da compra | KPIs do dia em colunas **aditivas** (contagens e somas), para recompor qualquer razão |
| `gold_atraso_vs_review` | faixa de atraso | a nota da review cai quando o pedido atrasa? |

**Regra do scorecard (opção A, ADR 0008):** um pedido pode ter itens de mais de um vendedor (1.256 pedidos). O pedido entra no scorecard de **cada** vendedor que tem item nele, com o status, o atraso e a review do pedido inteiro. Por isso `SUM(pedidos)` do scorecard (99.049) passa do total de pedidos com item (97.731); `pedidos_compartilhados` mostra quantos são desse tipo. O GMV não duplica: soma só os itens do próprio vendedor.

**Este arquivo é fonte de pipeline, não notebook comum** (roteiro na última célula).

```
fact_order_items ─┐
fact_orders ──────┼─► gold_seller_scorecard (por versão do vendedor, via dim_seller)
dim_seller ───────┘
fact_orders + dim_customer ─► gold_delivery_sla
fact_orders ───────────────► gold_daily_kpis · gold_atraso_vs_review
```

## Configuração

Mesmos imports e atalhos do `01_gold_modelo` (cada fonte tem o seu código Python, então eles são declarados de novo).

In [ ]:
# O QUE FAZ: importa o módulo do pipeline e as funções de coluna, lê o catálogo da configuração do
#   pipeline e declara o atalho de leitura de tabelas gold. Nenhum dado é lido aqui.
#
# FUNÇÕES DECLARADAS:
# - gold(nome): lê olist.gold.<nome>. Ler uma tabela do próprio pipeline cria a dependência: o pipeline
#   calcula fact_orders antes dos agregados que a leem.
#
# FUNÇÕES NATIVAS:
# - spark.conf.get(chave, padrão): lê um valor da configuração do pipeline ("olist" se a chave não existe).
# - spark.read.table(nome): lê uma tabela do Unity Catalog como DataFrame.
from pyspark import pipelines as dp
from pyspark.sql import functions as F

CATALOG = spark.conf.get("olist.catalogo", "olist")


def gold(nome: str):
    return spark.read.table(f"{CATALOG}.gold.{nome}")

## `gold_seller_scorecard`: uma linha por versão do vendedor

O agrupamento é feito em dois passos:

1. **Vendedor × pedido** (`itens_do_pedido`): dentro de cada pedido, quantos itens e quanto GMV pertencem àquela versão do vendedor. É aqui que o GMV fica só do vendedor, mesmo num pedido compartilhado.
2. **Versão do vendedor**: junta o status do pedido (`fact_orders`) e soma. As flags booleanas viram 0/1 com `cast("int")` para poderem ser somadas; `entregue_no_prazo` é nula para quem não foi entregue, e a soma ignora nulos.

Taxas ficam ao lado das contagens (`pedidos_cancelados`, `pedidos_entregues`, `entregues_no_prazo`) para quem quiser reagregar. A cidade, a UF e o intervalo de validade vêm de `dim_seller`.

In [ ]:
# O QUE FAZ: declara a materialized view gold_seller_scorecard (uma linha por versão do cadastro do
#   vendedor que vendeu), com volume, cancelamento, entrega e review, e as junta ao cadastro da versão.
#
# FUNÇÕES DECLARADAS:
# - gold_seller_scorecard(): devolve o DataFrame da materialized view. cluster_by pede liquid clustering
#   por vendedor (ganho educativo; ajuste fino na Fase 8).
#
# FUNÇÕES NATIVAS:
# - groupBy(...).agg(...): agrega por grupo. F.count("*") conta linhas, F.sum soma, F.avg tira a média
#   (ignora nulos), F.count(coluna) conta valores não nulos.
# - F.when(cond, valor): condicional; sem .otherwise o resultado é nulo quando a condição é falsa.
# - F.col("flag").cast("int"): True vira 1 e False vira 0 (nulo continua nulo), para poder somar.
# - F.coalesce(a, b): primeiro valor não nulo (GMV nulo de quem só vendeu em pedido inválido vira 0).
# - DataFrame.join(outro, "chave"): junção pela coluna de mesmo nome. .select(...): escolhe e renomeia.
# - F.lit(0): valor constante. F.col("a") / F.col("b"): divisão de colunas; F.when evita dividir por zero.
@dp.materialized_view(
    name=f"{CATALOG}.gold.gold_seller_scorecard",
    comment=(
        "Scorecard de vendedores: uma linha por versão do cadastro (seller_sk), com volume, GMV, cancelamento, "
        "entrega no prazo e review. Um pedido entra em cada vendedor que tem item nele (pedidos_compartilhados "
        "conta os que têm mais de um vendedor); o GMV soma só os itens do próprio vendedor. A cidade e a UF são "
        "as da data da venda."
    ),
    cluster_by=["seller_id"],
)
def gold_seller_scorecard():
    itens_do_pedido = (
        gold("fact_order_items")
        .groupBy("seller_sk", "order_id")
        .agg(
            F.count("*").alias("itens"),
            F.sum(F.when(F.col("pedido_valido"), F.lit(1)).otherwise(F.lit(0))).alias("itens_validos"),
            F.sum(F.when(F.col("pedido_valido"), F.col("item_total"))).alias("gmv"),
        )
    )
    pedidos = gold("fact_orders").select(
        "order_id", "pedido_valido", "cancelado", "entregue", "entregue_no_prazo",
        "atraso_dias", "lead_time_dias", "review_score", "qtd_vendedores",
    )
    por_versao = (
        itens_do_pedido.join(pedidos, "order_id")
        .groupBy("seller_sk")
        .agg(
            F.count("*").alias("pedidos"),
            F.sum(F.col("pedido_valido").cast("int")).alias("pedidos_validos"),
            F.sum((F.col("qtd_vendedores") > 1).cast("int")).alias("pedidos_compartilhados"),
            F.sum("itens").alias("itens"),
            F.sum("itens_validos").alias("itens_validos"),
            F.coalesce(F.sum("gmv"), F.lit(0)).alias("gmv"),
            F.sum(F.col("cancelado").cast("int")).alias("pedidos_cancelados"),
            F.sum(F.col("entregue").cast("int")).alias("pedidos_entregues"),
            F.sum(F.col("entregue_no_prazo").cast("int")).alias("entregues_no_prazo"),
            F.avg("atraso_dias").alias("atraso_medio_dias"),
            F.avg("lead_time_dias").alias("lead_time_medio_dias"),
            F.count(F.when(F.col("entregue"), F.col("review_score"))).alias("pedidos_com_review"),
            F.avg(F.when(F.col("entregue"), F.col("review_score"))).alias("review_score_medio"),
        )
        .withColumn("taxa_cancelamento", F.col("pedidos_cancelados") / F.col("pedidos"))
        .withColumn(
            "pct_entregas_no_prazo",
            F.when(F.col("pedidos_entregues") > 0, F.col("entregues_no_prazo") / F.col("pedidos_entregues")),
        )
    )
    vendedor = gold("dim_seller").select(
        "seller_sk", "seller_id", "seller_city", "seller_state", "seller_zip_code_prefix",
        "valido_de", "valido_ate", "versao_vigente", "cadastro_excluido_hoje",
    )
    return por_versao.join(vendedor, "seller_sk").select(
        "seller_sk", "seller_id", "seller_city", "seller_state", "seller_zip_code_prefix",
        "valido_de", "valido_ate", "versao_vigente", "cadastro_excluido_hoje",
        "pedidos", "pedidos_validos", "pedidos_compartilhados", "itens", "itens_validos", "gmv",
        "pedidos_cancelados", "taxa_cancelamento", "pedidos_entregues", "entregues_no_prazo",
        "pct_entregas_no_prazo", "atraso_medio_dias", "lead_time_medio_dias",
        "pedidos_com_review", "review_score_medio",
    )

## `gold_delivery_sla` e `gold_daily_kpis`

**`gold_delivery_sla`** (mês da compra × UF do cliente na compra) só olha pedidos **entregues**, como o KPI de prazo. Guarda contagens (entregues, no prazo, atrasados) e as medidas de atraso: `entregues_atrasados` é `atraso_dias > 0`, `atraso_medio_dos_atrasados_dias` mede só quem atrasou (a média de todos mistura atraso com antecipação).

**`gold_daily_kpis`** (dia da compra) guarda só **contagens e somas**. Uma média diária não pode ser somada para virar média mensal (a média das médias pesa um dia de 10 pedidos igual a um de 1.000); com `soma_lead_time_dias ÷ pedidos_entregues` qualquer período sai certo.

In [ ]:
# O QUE FAZ: declara duas materialized views: o SLA de entrega por mês e UF, e os KPIs diários aditivos.
#
# FUNÇÕES DECLARADAS:
# - gold_delivery_sla(): pedidos entregues agrupados por mês da compra e UF do cliente na compra.
# - gold_daily_kpis(): todos os pedidos agrupados pelo dia da compra.
#
# FUNÇÕES NATIVAS:
# - DataFrame.where("entregue"): só pedidos entregues (a coluna já é booleana).
# - F.date_format(data, "yyyy-MM"): mês da compra como texto.
# - F.expr("percentile(lead_time_dias, 0.5)"): mediana exata, escrita em SQL dentro do agg.
# - F.max / F.sum / F.avg / F.count: agregações. (F.col("atraso_dias") > 0).cast("int"): 1 se atrasou.
# - F.when(cond, valor): média só dos atrasados, porque o resto fica nulo e F.avg ignora nulos.
# - withColumn("pct_no_prazo", a / b): acrescenta a razão depois da agregação.
@dp.materialized_view(
    name=f"{CATALOG}.gold.gold_delivery_sla",
    comment=(
        "SLA de entrega por mês da compra e UF do cliente na compra, só pedidos entregues: no prazo, atrasados, "
        "lead time (média e mediana) e atraso. Contagens guardadas para reagregar."
    ),
)
def gold_delivery_sla():
    cliente = gold("dim_customer").select("customer_sk", F.col("customer_state").alias("uf_cliente"))
    entregues = gold("fact_orders").where("entregue").join(cliente, "customer_sk")
    return (
        entregues.groupBy(F.date_format("order_purchase_date", "yyyy-MM").alias("ano_mes"), "uf_cliente")
        .agg(
            F.count("*").alias("pedidos_entregues"),
            F.sum(F.col("entregue_no_prazo").cast("int")).alias("entregues_no_prazo"),
            F.sum((F.col("atraso_dias") > 0).cast("int")).alias("entregues_atrasados"),
            F.avg("lead_time_dias").alias("lead_time_medio_dias"),
            F.expr("percentile(lead_time_dias, 0.5)").alias("lead_time_mediana_dias"),
            F.avg(F.when(F.col("atraso_dias") > 0, F.col("atraso_dias"))).alias("atraso_medio_dos_atrasados_dias"),
            F.max("atraso_dias").alias("atraso_maximo_dias"),
        )
        .withColumn("pct_no_prazo", F.col("entregues_no_prazo") / F.col("pedidos_entregues"))
    )


@dp.materialized_view(
    name=f"{CATALOG}.gold.gold_daily_kpis",
    comment=(
        "KPIs por dia da compra em colunas aditivas (contagens e somas). Razões se calculam somando e dividindo "
        "depois: ticket = gmv / pedidos_validos; lead time = soma_lead_time_dias / pedidos_entregues."
    ),
    cluster_by=["order_purchase_date"],
)
def gold_daily_kpis():
    return (
        gold("fact_orders")
        .groupBy("order_purchase_date")
        .agg(
            F.count("*").alias("pedidos_criados"),
            F.sum(F.col("pedido_valido").cast("int")).alias("pedidos_validos"),
            F.coalesce(F.sum(F.when(F.col("pedido_valido"), F.col("gmv_bruto"))), F.lit(0)).alias("gmv"),
            F.sum(F.col("cancelado").cast("int")).alias("pedidos_cancelados"),
            F.sum(F.col("entregue").cast("int")).alias("pedidos_entregues"),
            F.sum(F.col("entregue_no_prazo").cast("int")).alias("entregues_no_prazo"),
            F.coalesce(F.sum("lead_time_dias"), F.lit(0.0)).alias("soma_lead_time_dias"),
            F.count(F.when(F.col("entregue"), F.col("review_score"))).alias("reviews_de_entregues"),
            F.coalesce(F.sum(F.when(F.col("entregue"), F.col("review_score"))), F.lit(0)).alias("soma_review_score"),
        )
    )

## `gold_atraso_vs_review`

Responde à dor do atendimento: **a nota cai quando o pedido atrasa?** Só entra pedido entregue **com** review (91.906). O atraso é `DATE(entrega) − DATE(entrega estimada)`; as faixas têm um número na frente para ordenar sozinhas. Cada faixa guarda quantos pedidos, a nota média e a parcela de notas baixas (1 ou 2) e de notas máximas (5).

In [ ]:
# O QUE FAZ: declara a materialized view gold_atraso_vs_review: nota média da review por faixa de atraso.
#
# FUNÇÕES DECLARADAS:
# - gold_atraso_vs_review(): pedidos entregues com review agrupados por faixa de atraso.
#
# FUNÇÕES NATIVAS:
# - F.col("a") & F.col("b").isNotNull(): "e" lógico; isNotNull() testa não nulo.
# - F.when(c1, v1).when(c2, v2)...otherwise(v): CASE WHEN em cadeia; vale o primeiro que for verdadeiro.
# - F.sum((F.col("review_score") <= 2).cast("int")): quantos pedidos têm nota 1 ou 2.
# - F.avg / F.count / withColumn: média, contagem e colunas de razão calculadas depois.
@dp.materialized_view(
    name=f"{CATALOG}.gold.gold_atraso_vs_review",
    comment=(
        "Nota média da review por faixa de atraso na entrega (atraso = data da entrega - data estimada), só "
        "pedidos entregues com review. Mostra se a nota cai quando o pedido atrasa."
    ),
)
def gold_atraso_vs_review():
    atraso = F.col("atraso_dias")
    faixa = (
        F.when(atraso <= -1, "1_adiantado")
        .when(atraso == 0, "2_no_dia")
        .when(atraso <= 3, "3_atraso_1_a_3_dias")
        .when(atraso <= 7, "4_atraso_4_a_7_dias")
        .when(atraso <= 14, "5_atraso_8_a_14_dias")
        .otherwise("6_atraso_15_dias_ou_mais")
    )
    base = gold("fact_orders").where(F.col("entregue") & F.col("review_score").isNotNull())
    return (
        base.groupBy(faixa.alias("faixa_atraso"))
        .agg(
            F.count("*").alias("pedidos"),
            F.avg("review_score").alias("review_score_medio"),
            F.sum((F.col("review_score") <= 2).cast("int")).alias("pedidos_nota_1_ou_2"),
            F.sum((F.col("review_score") == 5).cast("int")).alias("pedidos_nota_5"),
            F.avg("atraso_dias").alias("atraso_medio_dias"),
        )
        .withColumn("pct_nota_1_ou_2", F.col("pedidos_nota_1_ou_2") / F.col("pedidos"))
        .withColumn("pct_nota_5", F.col("pedidos_nota_5") / F.col("pedidos"))
    )

## Como executar e conferir

Este bloco é só roteiro, não é código do pipeline.

1. *Settings → Source code* do pipeline `olist.gold`: **adicione** `src/gold/03_gold_agregados` (o `01_gold_modelo` continua na lista). Não mexa no catálogo nem no schema do pipeline.
2. *Dry run*: o grafo deve mostrar 12 tabelas, com os quatro agregados novos ligados a `fact_orders` (o scorecard também a `fact_order_items` e `dim_seller`; o SLA também a `dim_customer`).
3. *Run pipeline* normal: só os quatro agregados são calculados; as outras 8 ficam `NO_OP`.

```sql
-- scorecard: uma linha por versão que vendeu
SELECT count(*) AS versoes, count(DISTINCT seller_id) AS vendedores,
       sum(pedidos) AS pares_pedido_vendedor, sum(itens) AS itens, sum(gmv) AS gmv
FROM olist.gold.gold_seller_scorecard;       -- 3.xxx · 3.077 · 99.049 · 111.592 · 15.614.632,11

-- o vendedor que se mudou: as vendas ficam na cidade que ele tinha na data da venda
SELECT seller_id, seller_city, seller_state, valido_de, valido_ate, itens, gmv, pct_entregas_no_prazo, review_score_medio
FROM olist.gold.gold_seller_scorecard
WHERE seller_id IN (SELECT seller_id FROM olist.gold.gold_seller_scorecard GROUP BY 1 HAVING count(*) > 1 LIMIT 1)
ORDER BY valido_de;

-- critério 6: itens vendidos antes e depois da primeira mudança de endereço (instante da compra)
WITH v AS (SELECT *, row_number() OVER (PARTITION BY seller_id ORDER BY valido_de) AS n,
                  count(*) OVER (PARTITION BY seller_id) AS versoes FROM olist.gold.gold_seller_scorecard)
SELECT n = 1 AS antes_da_mudanca, count(DISTINCT seller_id) AS vendedores, sum(itens) AS itens
FROM v WHERE versoes > 1 GROUP BY 1;         -- antes 102 · 4.022 | depois 213 · 7.231

-- atraso x review
SELECT faixa_atraso, pedidos, round(review_score_medio, 2) AS nota, round(pct_nota_1_ou_2, 3) AS pct_nota_baixa
FROM olist.gold.gold_atraso_vs_review ORDER BY faixa_atraso;

SELECT sum(pedidos_criados), sum(gmv), sum(pedidos_entregues) FROM olist.gold.gold_daily_kpis;   -- 98.476 · 15.614.632,11 · 93.247
SELECT sum(pedidos_entregues), sum(entregues_no_prazo) FROM olist.gold.gold_delivery_sla;        -- 93.247 · 86.974
```

**Sobre o critério 6 do desenho:** a Fase 3 contou 3.906 itens antes e 7.347 depois da mudança, usando o limite de envio (`shipping_limit_date`). A gold usa o **instante da compra**, que é a regra do projeto: 4.022 antes e 7.231 depois (o total, 11.253, é o mesmo). Os 116 itens de diferença foram comprados antes da mudança, mas o limite de envio caiu depois dela.